In [1]:
from pathlib import Path
import time, gc, warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "train"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "model_results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(FEATURE_DIR / "train_candidates.csv")
FEATURES = pd.read_csv(FEATURE_DIR / "feature_columns.csv")["feature"].tolist()
train["trip_id"] = train.trip_id.astype(str).str.zfill(8)
train["target"] = train.target.astype(int)
group_cols = ["trip_id", "observation_pos"]
# Remove observations for which the true arc was not in the generated candidate set.
valid_obs = train.groupby(group_cols).target.transform("max") == 1
train = train.loc[valid_obs].copy()
train[FEATURES] = train[FEATURES].replace([np.inf, -np.inf], np.nan)
print("Rows:", len(train), "Trips:", train.trip_id.nunique())
print("Target counts:", train.target.value_counts().to_dict())
print("Features:", FEATURES)

Rows: 2590102 Trips: 80
Target counts: {0: 2384060, 1: 206042}
Features: ['candidate_rank', 'distance_m', 'distance_to_projected_point_m', 'segment_length_m', 'gps_heading_deg', 'road_heading_deg', 'heading_difference_deg', 'projection_fraction', 'used_fallback', 'search_radius_m']


In [2]:
try:
    import optuna
except ImportError:
    %pip install optuna
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_SPLITS = 3
N_TRIALS = 12       # Increase only if time permits
TIMEOUT_SECONDS = 900
MAX_FIT_ROWS = 150_000
RANDOM_STATE = 42

# Check whether this XGBoost build can actually use CUDA.
USE_CUDA = True
try:
    _probe = XGBClassifier(n_estimators=2, tree_method="hist", device="cuda",
                           eval_metric="logloss", n_jobs=2)
    _probe.fit(np.array([[0.],[1.],[2.],[3.]]), np.array([0,0,1,1]))
    del _probe
    print("XGBoost CUDA is available.")
except Exception as e:
    USE_CUDA = False
    print("XGBoost CUDA unavailable; using CPU.", str(e)[:250])

def sampled_fit_data(frame, max_rows=MAX_FIT_ROWS, seed=RANDOM_STATE):
    if len(frame) <= max_rows:
        return frame
    pos = frame[frame.target == 1]
    neg = frame[frame.target == 0]
    if len(pos) == 0 or len(neg) == 0:
        raise ValueError("A training fold contains only one target class.")
    n_pos = min(len(pos), max_rows // 2)
    n_neg = min(len(neg), max_rows - n_pos)
    if n_pos + n_neg < max_rows:
        extra = max_rows - n_pos - n_neg
        add = min(extra, len(pos) - n_pos); n_pos += add; extra -= add
        if extra: n_neg += min(extra, len(neg) - n_neg)
    return pd.concat([pos.sample(n_pos, random_state=seed),
                      neg.sample(n_neg, random_state=seed)]).sample(frac=1, random_state=seed)

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(train, train.target, groups=train.trip_id))
for i, (tr, va) in enumerate(folds, 1):
    print(f"Fold {i}: train classes={train.iloc[tr].target.nunique()}, validation trips={train.iloc[va].trip_id.nunique()}")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XGBoost CUDA is available.
Fold 1: train classes=2, validation trips=26
Fold 2: train classes=2, validation trips=27
Fold 3: train classes=2, validation trips=27


In [3]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 220, step=20),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    scores = []
    for fold_no, (tr_idx, va_idx) in enumerate(folds, 1):
        tr = sampled_fit_data(train.iloc[tr_idx], seed=RANDOM_STATE + fold_no + trial.number)
        va = train.iloc[va_idx]
        model = XGBClassifier(
            **params, objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", device="cuda" if USE_CUDA else "cpu",
            n_jobs=2, random_state=RANDOM_STATE,
        )
        model.fit(tr[FEATURES], tr.target)
        pred = (model.predict_proba(va[FEATURES])[:, 1] >= 0.5).astype(int)
        scores.append(f1_score(va.target, pred, average="macro", zero_division=0))
        del model, tr, va, pred
        gc.collect()
        trial.report(float(np.mean(scores)), fold_no)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=1))
start = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS, gc_after_trial=True, show_progress_bar=True)
search_seconds = time.perf_counter() - start
print("Completed trials:", len(study.trials), "Search time (s):", round(search_seconds, 1))
print("Best CV Macro F1:", round(study.best_value, 5))
print("Best parameters:", study.best_params)

  0%|          | 0/12 [00:00<?, ?it/s]c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:569: UserWarning: [15:31:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Best trial: 11. Best value: 0.811698: 100%|██████████| 12/12 [00:31<00:00,  2.66s/it, 31.88/900 seconds]

Completed trials: 12 Search time (s): 31.9
Best CV Macro F1: 0.8117
Best parameters: {'n_estimators': 120, 'max_depth': 8, 'learning_rate': 0.11892927632302643, 'subsample': 0.7751670433795944, 'colsample_bytree': 0.7362708966123857, 'min_child_weight': 1.4772756085620753, 'reg_lambda': 0.04886613239167845, 'reg_alpha': 0.9811170143048075, 'gamma': 3.2498134487592147}


In [4]:
best_params = dict(study.best_params)
final_fit = sampled_fit_data(train, seed=RANDOM_STATE)
best_model = XGBClassifier(
    **best_params, objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", device="cuda" if USE_CUDA else "cpu",
    n_jobs=2, random_state=RANDOM_STATE,
)
start = time.perf_counter()
best_model.fit(final_fit[FEATURES], final_fit.target)
fit_seconds = time.perf_counter() - start

model_path = MODEL_DIR / "best_candidate_model_bayesian_xgb.joblib"
joblib.dump(best_model, model_path)
(OUTPUT_DIR / "bayesian_best_params.json").write_text(__import__("json").dumps(best_params, indent=2), encoding="utf-8")
(OUTPUT_DIR / "bayesian_optimization_summary.csv")
pd.DataFrame([{"model":"XGBoost (Bayesian optimised)", "cv_macro_f1":study.best_value,
               "search_seconds":search_seconds, "final_fit_seconds":fit_seconds,
               "device":"cuda" if USE_CUDA else "cpu", "trials_completed":len(study.trials)}]).to_csv(OUTPUT_DIR / "bayesian_optimization_summary.csv", index=False)

trials_df = study.trials_dataframe()
trials_df.to_csv(OUTPUT_DIR / "bayesian_optimization_trials.csv", index=False)
print("Saved model:", model_path)
print("Saved parameters:", OUTPUT_DIR / "bayesian_best_params.json")
print("Saved trial history:", OUTPUT_DIR / "bayesian_optimization_trials.csv")
display(pd.read_csv(OUTPUT_DIR / "bayesian_optimization_summary.csv"))

Saved model: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\models\best_candidate_model_bayesian_xgb.joblib
Saved parameters: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\model_results\bayesian_best_params.json
Saved trial history: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\model_results\bayesian_optimization_trials.csv


,model,cv_macro_f1,search_seconds,final_fit_seconds,device,trials_completed
0,XGBoost (Bayesian optimised),0.811698,31.887494,0.440901,cuda,12
